# Extruder puffs: finding and measuring each piece at the moment it is cut

**CIS 831 Deep Learning project, Kansas State University**

This notebook takes one slow-motion video of the die and, for **every cut of the knife**:

1. finds the moment a fresh rope end comes out of **die hole A**;
2. lets **SAM 2.1** (a pretrained AI model from Meta that outlines objects in images and videos)
   follow that piece frame by frame;
3. marks the **cut moment** (RED, the last frame the piece still touches the hole) and the
   **first detached frame** (BLUE);
4. saves both outlines (*masks*) and measures the piece in **millimetres**, using the bolts
   on the die plate as a ruler.

The only manual step is clicking hole A and the two bolts **once per video**.

### How to run it (VS Code)
1. Open this notebook and select the kernel **Python (extruder-sam2)** (top right).
2. Run the cells **one at a time** from the top (Shift+Enter).
3. Stop at **Step 4 (click the points)**. Click, press **Save points**, then continue.
4. Results are written to `results/<video name>/`.

### Words used in this notebook
| Word | Meaning |
|---|---|
| **frame** | one picture of the video (our camera takes about 240 per second) |
| **crop** | a smaller square cut out of the frame around hole A (a digital zoom) |
| **mask** | a black-and-white picture: white where the piece is, black elsewhere |
| **prompt** | how we tell SAM 2 which object we mean (here: a box drawn around it) |
| **tracking** | SAM 2 following the same object through the next frames |
| **event** | the frame where a new rope end starts to come out (one per knife cut) |
| **head-on frame** | the frame after the event, when the new rope end looks like a small round disc |

## Step 1: Imports
Loads the tools we use. `extruder` is this project's own helper code (folder `src/extruder/`);
every function there has a plain-language explanation at the top of its file.

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Image as ShowImage

from extruder import video as vid, cuts, calib, prompt, track, measure, pipeline, viz
from extruder.ui import ReferenceClicker

%matplotlib inline
plt.rcParams["figure.dpi"] = 90
print("PyTorch", torch.__version__, "| GPU available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Step 2: Configuration
Everything you may want to change is in this one cell.

* `VIDEO`: the file name of one video inside `data/`. Set it to `None` to see a list of all
  videos in `data/` and pick one by number with `VIDEO_INDEX`.
* `MAX_CUTS`: `None` processes every cut. Set it to a small number (for example `20`) for a quick try.
* `N_TRACK` and `CALIBRATION_POINTS`: leave them as `None` (automatic).
* `MANUAL_POINTS`: leave it as `None` to click in Step 4. You can also type the points here,
  for example `{"hole_A": (723, 487), "bolt_near": (690, 533), "bolt_far": (854, 379)}`.

In [ ]:
# ---- folders ------------------------------------------------------------------
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"          # put the videos here
RESULTS_DIR = PROJECT_DIR / "results"    # everything we produce goes here

# ---- which video ----------------------------------------------------------------
VIDEO = "DJI_20261007131306_0069_D.MP4"   # a file name in data/, or None to choose from the list
VIDEO_INDEX = 0                           # used only when VIDEO is None

# ---- SAM 2.1 model ------------------------------------------------------------------
SAM2_CHECKPOINT = PROJECT_DIR / "checkpoints" / "sam2.1_hiera_large.pt"   # the trained weights (downloaded by setup_env.sh)
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"                       # the matching model description (ships with SAM 2)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- pipeline settings (the values used in all our tests) ------------------------------
CROP_SIZE = 360       # side of the square crop around hole A, in pixels
N_TRACK = None        # frames SAM 2 follows each piece; None = automatic (about 1.6 knife cycles)
MAX_CUTS = None       # None = all cuts in the video; a number = only the first N cuts
EXAMPLE_CUT = None    # which cut to show step by step; None = pick the first clean one automatically

# ---- reference points (pixels -> millimetres) ---------------------------------------------
MANUAL_POINTS = None                            # None = click them in Step 4
CALIBRATION_POINTS = None   # points that set the mm scale; None = automatic (see Step 5)
USE_BRIGHTNESS = False      # True = SAM 2 sees the video at the brightness saved in Step 4

videos = vid.list_videos(DATA_DIR)
print(f"Videos in {DATA_DIR}:")
for i, v in enumerate(videos):
    print(f"  [{i}] {v.name}")
VIDEO_PATH = DATA_DIR / VIDEO if VIDEO else videos[VIDEO_INDEX]
assert VIDEO_PATH.exists(), f"Video not found: {VIDEO_PATH}. Copy your videos into {DATA_DIR}."
OUT = RESULTS_DIR / VIDEO_PATH.stem
(OUT / "figures").mkdir(parents=True, exist_ok=True)
print("\nWorking on:", VIDEO_PATH.name, "\nResults go to:", OUT)

## Step 3: Look at the video
Basic facts about the file, and one full frame as the camera sees it.

In [ ]:
info = vid.probe(VIDEO_PATH)
display(pd.Series({"frames per second": round(info["fps"], 2), "number of frames": info["n_frames"],
                   "length (s)": info["duration_s"], "size (px)": f"{info['width']} x {info['height']}",
                   "video format": f"{info['codec']} ({info['pixel_format']})"}).to_frame("value"))

some_frame = vid.read_frame(VIDEO_PATH, info["n_frames"] // 2, info["fps"], info["width"], info["height"])
plt.figure(figsize=(13, 7.5))
plt.imshow(some_frame)
plt.title(f"Frame {info['n_frames'] // 2} of {info['n_frames']}: the whole camera view")
plt.axis("off")
plt.show()

### The "clean die" picture
For each pixel we take the **median** over about 70 frames spread over the clip. The die plate
does not move but the product does, so the moving pieces mostly disappear and the **bolts
become easy to see**. We use this picture for clicking. It takes about 15 s and is saved
for next time.

In [ ]:
clean_path = OUT / "clean_die.png"
if clean_path.exists():
    clean = np.asarray(Image.open(clean_path).convert("RGB"))
else:
    clean = vid.clean_die_image(VIDEO_PATH)
    Image.fromarray(clean).save(clean_path)

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
for a, img, t in ((ax[0], some_frame, "One frame"), (ax[1], clean, "Clean die (median of many frames)")):
    a.imshow(viz.brighten(img[230:750, 520:1060], 1.6)); a.set_title(t + ", brightened, around the die"); a.axis("off")
plt.show()

## Step 4: Click hole A and the reference points
An interactive picture appears below.

1. In **Point**, choose what you are about to click, then **left-click** it in the picture:
   * **hole A**: the die hole we follow (the lower one, where the yellow product comes out);
   * **bolt just below hole A**: click the **centre of the bolt head**;
   * **other bolt**: centre of its head;
   * **hole B**: optional (see Step 5).
2. To click precisely, use the **magnifier** in the figure's toolbar to zoom in. Click the
   magnifier again to switch zoom off before clicking a point (clicks are ignored while
   zoom is on). The **house** button resets the view.
3. **View** switches between the clean-die picture and single frames (use the slider).
   **Brightness** (or **Auto brightness**) helps in dark videos; it is saved with the points. SAM 2 uses it only
   if `USE_BRIGHTNESS = True` in Step 2. It is off by default: on our darkest video it did not give cleaner masks,
   and it made them about 3 % larger.
4. Press **Save points**. They are stored in `results/<video>/clicks.json` and shown again
   next time, so you only click once per video.

The hole A click does not need to be exact: the computer moves it to the exact hole centre
in Step 5.

In [ ]:
saved = pipeline.load_clicks(OUT / "clicks.json")
start_points = MANUAL_POINTS or saved.get("clicked") or {}

%matplotlib widget
clicker = ReferenceClicker(VIDEO_PATH, info, clean, OUT / "clicks.json", points=start_points,
                           brightness=saved.get("brightness", 1.0))
clicker.show()

## Step 5: Exact hole centres, and the millimetre ruler
**Holes A and B:** the computer reads a small window around each click for the whole clip and finds
every fresh round rope end. The median of their centres is the hole centre (clicks up to ~25 px off
all end at the same centre).

**Millimetres:** the lab measured the die plate on the cold die:

| | mm |
|---|---|
| die hole diameter (A and B) | 4.28 |
| bolt head diameter | 14 |
| plate centre to die holes / to bolts | 31 / 45 |
| hole A to hole B | 55 |
| hole A to the bolt just below it | 16 |
| bolt to bolt | 65 |

From these we draw a map of the plate in mm and match it to the clicked points. Which points set the
scale is chosen automatically:

1. **Both bolts visible → the bolts** (65 mm apart). They are true plate points.
2. **Otherwise holes A and B** (55 mm apart). Both are found from their rope ends. The ropes stick out
   toward the camera by the same amount, so that shift cancels in the distance between the holes.
   This is the case in the trial videos, where only one bolt is visible.

Every other clicked point is a **check**: the table and figure show how far it is from where the map
puts it.

In [ ]:
%matplotlib inline
plt.close(clicker.fig)   # the click picture is not needed any more (re-run Step 4 to change points)
clicked = {k: v for k, v in clicker.points.items() if v is not None}
if "hole_A" not in clicked:
    print("Hole A is missing - go back to Step 4.")

refined, hole_info = pipeline.refine_points(VIDEO_PATH, info, clicked)
points = {**clicked, **refined}
use, ref_name = (CALIBRATION_POINTS, "chosen in Step 2") if CALIBRATION_POINTS else calib.choose_reference(points)
if use:
    mapping = calib.fit_mapping(points, use=use)
    PX_TO_MM = mapping["px_to_mm"]
else:
    mapping, PX_TO_MM = None, None
    print("No millimetre conversion: click both bolts, or holes A and B (Step 4).")

BRIGHTNESS = clicker.brightness if USE_BRIGHTNESS else 1.0
json.dump({"video": VIDEO_PATH.name, "clicked": clicked, "refined": {k: list(v) for k, v in refined.items()},
           "brightness": clicker.brightness,
           "calibration_points": list(use) if use else None,
           "px_per_mm": mapping["px_per_mm"] if mapping else None},
          open(OUT / "clicks.json", "w"), indent=1)

if mapping:
    print(f"\nMapping: {mapping['kind']} from {mapping['points_used']} ({ref_name})")
    print(f"Scale: {mapping['px_per_mm']:.3f} px per mm  (1 px = {1 / mapping['px_per_mm']:.3f} mm)")
    display(pd.Series(mapping["offset_mm"]).round(2).to_frame("distance between point and map (mm)"))
    viz.calibration_figure(clean, clicked, refined, mapping, path=OUT / "figures" / "calibration.png")
    plt.show()

## Step 6: The crop (digital zoom) and finding the cuts
We cut a **360 × 360 px square centred on hole A** out of every frame and save them as
pictures in `results/<video>/crop_frames/` (SAM 2 reads these).

While doing that, we measure the **blink**. A small circle on hole A turns dark when the
knife covers it, then yellow again when a new rope end comes out. The rhythm of this blink
gives the number of cuts per second without knowing the knife speed. Each blink gives one
**event**.

In [ ]:
prep = pipeline.prepare_crops(VIDEO_PATH, OUT, refined["hole_A"], CROP_SIZE, brightness=BRIGHTNESS)
HOLE = prep["hole_in_crop"]          # hole A inside the crop (close to 180, 180)
FILES = prep["frame_files"]          # one JPEG per video frame
events = prep["cuts"]["events"]
N_TRACK = N_TRACK or pipeline.adaptive_n_track(prep["cuts"]["period_frames"])
print(f"Tracking window: {N_TRACK} frames ({prep['cuts']['period_frames']:.1f} frames per knife cycle)")
print(f"Crop top-left corner in the frame: {prep['origin']}, hole A inside the crop: ({HOLE[0]:.1f}, {HOLE[1]:.1f})")

viz.signal_figure(prep["signal"], info["fps"], prep["cuts"], path=OUT / "figures" / "blink_signal.png")
plt.show()

### What the crop looks like
1. The full frame with the crop window.
2. The crop itself, with the windows the computer uses:
   * yellow dashed = the 80 px area used to pick the colour threshold;
   * orange = the furthest the automatic box may reach;
   * white = the 16 px **"is it still attached?" square**.
3. A close-up of hole A.

In [ ]:
example_event = int(events[len(events) // 2]) if EXAMPLE_CUT is None else int(events[EXAMPLE_CUT])
headon = vid.read_frame(VIDEO_PATH, example_event + 1, info["fps"], info["width"], info["height"])
viz.crop_explainer(headon, refined["hole_A"], prep["origin"], CROP_SIZE, path=OUT / "figures" / "crops.png")
plt.show()
viz.cycle_strip(FILES, example_event, HOLE, path=OUT / "figures" / "one_knife_cycle.png")
plt.show()

## Step 7: Load the SAM 2.1 model
**SAM 2** ("Segment Anything Model 2", Meta AI, 2024) is a large neural network trained on
millions of images and videos to outline any object it is pointed at. We use it **as it is,
without retraining**.

* `build_sam2_video_predictor` builds the network from its description (`SAM2_CONFIG`) and
  loads the trained weights (`SAM2_CHECKPOINT`, about 900 MB, the **large** version).
* The *video predictor* remembers the object from frame to frame, which is what lets it
  follow one piece.
* On a GPU it runs in `bfloat16` (a faster number format) where supported.

In [ ]:
from sam2.build_sam import build_sam2_video_predictor

predictor = build_sam2_video_predictor(SAM2_CONFIG, str(SAM2_CHECKPOINT), device=DEVICE)
track.quiet_sam2()   # hide SAM 2's per-cut progress bars

n_params = sum(p.numel() for p in predictor.parameters()) / 1e6
print(f"SAM 2.1 large loaded on {DEVICE}: {n_params:.0f} million parameters")

## Step 8: One cut, step by step
This is the whole method on one cut.

**8a. The automatic prompt (no click on the rope).** On the head-on frame:
1. a fresh rope end sits on hole A;
2. the **yellowness map** (red minus blue colour) makes the product bright and the steel dark;
3. the yellow blob on the hole is selected and the computer draws a **box** around it;
4. the box is given to SAM 2, as if a person had drawn it, and SAM 2 returns the mask.

In [ ]:
# pick the example cut: the one chosen above, or the next one that passes all checks
order = [int(np.where(events == example_event)[0][0])] + list(range(len(events)))
for k in order:
    row, res = pipeline.process_cut(predictor, prep, k, events[k], device=DEVICE, n_track=N_TRACK)
    if row["keep"] or EXAMPLE_CUT is not None:
        break
f0 = row["headon_frame"]
print(f"Example: cut {k}, event at frame {row['event_frame']}, head-on frame {f0}")

img0 = np.asarray(Image.open(FILES[f0]).convert("RGB"))
box, steps = prompt.disc_box(img0, HOLE, return_steps=True)
viz.prompt_steps_figure(img0, steps, res["masks"][f0], HOLE, path=OUT / "figures" / "example_prompt_steps.png")
plt.show()

**8b. SAM 2 follows the piece.** For every following frame we ask: *does the mask still
overlap the white square on hole A?*

* **green**: yes, the piece is attached;
* **RED**: the last attached frame, i.e. **the cut moment**;
* **BLUE**: the first frame where it has left the hole, i.e. **just detached**;
* grey: later frames, ignored. (SAM 2 sometimes jumps to the next rope end here; that does not
  matter because we stop at the first blue.)

The letters below the figure are the same story frame by frame: `A` = attached, `f` = free.

In [ ]:
viz.track_strip(FILES, res, f0, HOLE, path=OUT / "figures" / "example_tracking.png")
plt.show()
print("timeline:", row["timeline"], f"-> RED = frame {row['red_frame']}, BLUE = frame {row['blue_frame']}, "
      f"cut happened {row['frames_to_cut']} frames after the head-on frame")

**8c. The same cut as a slow animation** (saved as `figures/example_tracking.gif`).

In [ ]:
gif = viz.tracking_gif(FILES, res, f0, HOLE, OUT / "figures" / "example_tracking.gif")
display(ShowImage(filename=str(gif)))

**8d. Measuring the example piece.** The mask is redrawn on a millimetre grid using the bolts.
Both RED and BLUE are measured, the same way the lab measures the pieces.

* **Length:** like the lab's thread laid along the piece: along the line through the centres of the
  biggest circles that fit inside it, from tip to tip, following the bend.
* **Diameter:** along the piece's middle line, the biggest circle that fits inside the piece is the
  local thickness. The diameter is the median thickness over the central 60 % of the piece; the rounded
  ends are skipped.
* Each piece is measured at 4 rotations and the median is kept, so the direction it points does not matter.

| Measurement | Meaning |
|---|---|
| `length_mm` | **length along the bend**, tip to tip (like the lab's thread) |
| `width_mm` | **diameter**: median thickness over the central 60 % of the piece |
| `width_max_mm` | thickest point |
| `width_mean_mm` | area ÷ length |
| `feret_mm` | longest straight distance across the piece (reference only) |
| `area_mm2` | how much of the picture the piece covers |
| `straightness` | straight distance ÷ length along the bend (1 = straight) |
| `shape_stability_pct` | how much length/diameter change between the 4 rotations; above 5 % means an odd or ragged mask |
| `expansion` | (width ÷ 4.28 mm)², for reference only; the team computes the expansion ratio from the diameters |

Step 9b below shows, with figures, how these numbers are obtained and which masks are dropped.

In [ ]:
rows = {}
for key in ("red", "blue"):
    m = res["masks"][row[f"{key}_frame"]]
    rows[f"{key.upper()} (frame {row[f'{key}_frame']})"] = measure.measure_mask(m, prep["origin"], PX_TO_MM)
display(pd.DataFrame(rows).round(2))

## Step 9: All cuts in the video
Now the same steps for every cut. That is about 1 second per cut on a modern GPU, so about
4–5 minutes for a 10 s clip.

**Automatic checks:** a cut is **kept** only if all of these hold (otherwise the reason is
written in the table):

| Reason | Meaning |
|---|---|
| `no_rope_end_at_hole` | no yellow blob on the hole in the head-on frame |
| `no_detachment_within_window` | the piece never left the hole within 16 frames |
| `mask_lost_before_cut` | SAM 2 lost the piece just before it detached |
| `touches_crop_edge` | the piece runs out of the crop, so it cannot be measured whole |
| `too_small` | the RED mask is tiny (< 200 px) |
| `size_jump_red_to_blue` | the size more than doubles or halves between RED and BLUE (usually another piece merging in) |
| `unusual_size_for_this_clip` | very far from the clip's typical size |

For each cut, these files are saved:
* `masks/cut0007_red_f01762.png` and `masks/cut0007_blue_f01763.png` (cut number, colour, video frame number);
* the same names in `overlays/` as pictures, to check by eye.

In [ ]:
table = pipeline.run_cuts(predictor, prep, OUT, PX_TO_MM, device=DEVICE, n_track=N_TRACK, max_cuts=MAX_CUTS)
print(f"{table['keep'].sum()} of {len(table)} cuts kept")
display(table["reason"].replace("", "kept").value_counts().to_frame("cuts"))
show_cols = ["cut", "event_frame", "red_frame", "blue_frame", "frames_to_cut", "keep", "reason",
             "red_width_mm", "red_length_mm", "blue_width_mm", "blue_length_mm"]
display(table[[c for c in show_cols if c in table]].head(15).round(2))

## Step 9b: How a piece is measured, and which masks are dropped
The same explanations as `reports/how_we_measure_a_puff.pdf` and `reports/how_we_drop_bad_masks.pdf`,
on pieces of this video.

**Measuring** (8 panels):
1. the mask, then in millimetres;
2. the two tips, found by walking inside the piece;
3. the "thread" along the ridge of the biggest inside circles;
4. **length** = between the last full-size circles + one radius at each end;
5. **diameter** = median circle diameter over the central 60 %.

**Dropping:**
- **Check 1, ragged outline:** more than 2.5 % of the area changes when the outline is smoothed.
- **Check 2, one tube?** area ÷ (length × diameter) outside 0.80–1.20, i.e. a merged or a cut-off piece.
- **Check 3:** the 4 turned copies disagree by more than 5 %.

Dropped masks stay in `cuts.csv` with their reason (`red_mask_problem`, `blue_mask_problem`) but are left
out of the averages in Step 11.

In [ ]:
from extruder import explain
if PX_TO_MM is not None:
    used = table[table["keep"] & table["blue_mask_problem"].fillna("").eq("")]
    typical = used.iloc[(used["blue_length_mm"] - used["blue_length_mm"].median()).abs().argsort()[:10]]
    ex = int(typical.sample(1)["cut"].iloc[0])
    explain.measure_steps(OUT, ex, "blue", PX_TO_MM, path=OUT / "figures" / "how_one_piece_is_measured.png")
    plt.show()
else:
    print("No millimetre scale (Step 5), so the explanation figures are skipped.")

In [ ]:
if PX_TO_MM is not None:
    k = table[table["keep"]]
    ragged = k[k["blue_mask_problem"] == "ragged_outline"].sort_values("blue_roughness_pct", ascending=False)["cut"].head(2)
    odd = k[k["blue_mask_problem"].isin(["extra_area_merged_piece", "missing_area_cut_off_piece"])]["cut"].head(2)
    explain.roughness_check(OUT, [ex] + [int(c) for c in ragged], "blue", PX_TO_MM,
                            path=OUT / "figures" / "drop_check1_roughness.png"); plt.show()
    explain.tube_check(OUT, [ex] + [int(c) for c in odd], "blue", PX_TO_MM,
                       path=OUT / "figures" / "drop_check2_tube.png"); plt.show()
    long = pd.concat([k.rename(columns=lambda c: c.replace(f"{key}_", "")).assign(key=key, video=OUT.name)
                      for key in ("red", "blue")], ignore_index=True)
    long["mask_problem"] = long["mask_problem"].fillna("")
    for key in ("red", "blue"):
        explain.checks_histograms(long, key, title=f"{OUT.name}: drop scores of the {key.upper()} masks",
                                  path=OUT / "figures" / f"drop_scores_{key}.png"); plt.show()
    explain.dropped_gallery(long, RESULTS_DIR, OUT.name, "blue", path=OUT / "figures" / "dropped_blue.png"); plt.show()
    display(long.assign(result=long["mask_problem"].replace("", "used")).groupby(["key", "result"]).size().unstack(0))

## Step 10: Check the results by eye
Random kept cuts: **top row RED** (cut moment), **bottom row BLUE** (just detached).
Run the cell again to see other cuts.

In [ ]:
kept = table[table["keep"]].to_dict("records")
viz.gallery(FILES, random.sample(kept, min(8, len(kept))), HOLE, path=OUT / "figures" / "gallery.png")
plt.show()

## Step 11: Summary of this video: mean ± SD
The puffs are not uniform, so each video is described by the **mean and the spread
(standard deviation, SD)** of every measurement over all kept cuts whose masks passed the checks of Step 9b. RED and BLUE are reported separately.

In [ ]:
unit = "mm" if PX_TO_MM is not None else "px"
cols = [f"width_{unit}", f"length_{unit}", f"area_{unit}2"] + (["expansion"] if unit == "mm" else [])
viz.measurement_hists(table, cols, path=OUT / "figures" / "measurement_histograms.png")
plt.show()

summary = pipeline.save_summary(table, OUT, VIDEO_PATH.name,
                                dict(cut_hz=prep["cuts"]["cut_hz"],
                                     px_per_mm=mapping["px_per_mm"] if mapping else None,
                                     hole_A_px=list(refined["hole_A"]), crop_origin=list(prep["origin"])))
nice = {}
for m in pipeline.MEASURES:
    for key in ("red", "blue"):
        if f"{key}_{m}_mean" in summary:
            nice.setdefault(m, {})[key.upper()] = f"{summary[f'{key}_{m}_mean']:.2f} ± {summary[f'{key}_{m}_sd']:.2f}"
print(f"{summary['cuts_kept']} kept cuts of {summary['cuts_detected']}")
display(pd.DataFrame(nice).T)
print("Saved:", OUT / "cuts.csv", "and", OUT / "summary.json")

## Step 12 (optional): Run every video in `data/`
After you have clicked and saved the points for each video (Steps 2–5, changing `VIDEO`),
this cell processes all of them without further clicks. Videos without saved points are skipped.
It also writes `results/all_videos_summary.csv` (one row per video). Set `RUN_ALL = True` to use it.

In [ ]:
RUN_ALL = False
if RUN_ALL:
    rows = []
    for v in vid.list_videos(DATA_DIR):  # see notebook 02 for the full Experiment 1
        print("\n=====", v.name)
        s = pipeline.run_video(v, RESULTS_DIR, predictor, CROP_SIZE, CALIBRATION_POINTS, DEVICE, N_TRACK, MAX_CUTS)
        if s:
            rows.append(s)
    if rows:
        pd.DataFrame(rows).to_csv(RESULTS_DIR / "all_videos_summary.csv", index=False)
        display(pd.DataFrame(rows))

## Notes and limits
* **Validation still needed:** compare `width_mm` with caliper measurements of pieces collected from the same run,
  and hand-check a sample of masks in `overlays/`.
* **Steam and very dark videos** can make the yellowness blink weak; check the blink figure in Step 6.
* **Camera moved between videos:** click the points again for each video (Step 4).
* The lab confirmed that a piece keeps its size after the cut, so RED/BLUE sizes are directly comparable with the lab's measurements (seen from one side).